In [286]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

matches_path = "../data/matches_1930_2022.csv"

df = pd.read_csv(matches_path)

In [287]:
matches = df[["home_team", "away_team", "Year", "home_score", "away_score"]].copy()
matches

,home_team,away_team,Year,home_score,away_score
0,Argentina,France,2022,3,3
1,Croatia,Morocco,2022,2,1
2,France,Morocco,2022,2,0
3,Argentina,Croatia,2022,3,0
4,Morocco,Portugal,2022,1,0
...,...,...,...,...,...
959,Argentina,France,1930,1,0
960,Yugoslavia,Brazil,1930,2,1
961,Romania,Peru,1930,3,1
962,United States,Belgium,1930,3,0


In [288]:
home_matches = matches.copy()

home_matches = home_matches.rename(columns = {
    "home_team": "team",
    "away_team": "opponent",
    "Year": "year",
    "home_score": "goals_for",
    "away_score": "goals_against"
})

home_matches

,team,opponent,year,goals_for,goals_against
0,Argentina,France,2022,3,3
1,Croatia,Morocco,2022,2,1
2,France,Morocco,2022,2,0
3,Argentina,Croatia,2022,3,0
4,Morocco,Portugal,2022,1,0
...,...,...,...,...,...
959,Argentina,France,1930,1,0
960,Yugoslavia,Brazil,1930,2,1
961,Romania,Peru,1930,3,1
962,United States,Belgium,1930,3,0


In [289]:
away_matches = matches.copy()

away_matches = away_matches.rename(columns = {
    "away_team": "team",
    "home_team": "opponent",
    "Year": "year",
    "away_score": "goals_for",
    "home_score": "goals_against"
})

away_matches

,opponent,team,year,goals_against,goals_for
0,Argentina,France,2022,3,3
1,Croatia,Morocco,2022,2,1
2,France,Morocco,2022,2,0
3,Argentina,Croatia,2022,3,0
4,Morocco,Portugal,2022,1,0
...,...,...,...,...,...
959,Argentina,France,1930,1,0
960,Yugoslavia,Brazil,1930,2,1
961,Romania,Peru,1930,3,1
962,United States,Belgium,1930,3,0


In [290]:
def match_result(match):
    if match["goals_against"] < match["goals_for"]:
        return "W"
    elif match["goals_against"] == match["goals_for"]:
        return "D"
    else:
        return "L"


In [291]:
home_matches["result"] = home_matches.apply(match_result, axis=1)
away_matches["result"] = away_matches.apply(match_result, axis=1)

In [292]:
home_matches

,team,opponent,year,goals_for,goals_against,result
0,Argentina,France,2022,3,3,D
1,Croatia,Morocco,2022,2,1,W
2,France,Morocco,2022,2,0,W
3,Argentina,Croatia,2022,3,0,W
4,Morocco,Portugal,2022,1,0,W
...,...,...,...,...,...,...
959,Argentina,France,1930,1,0,W
960,Yugoslavia,Brazil,1930,2,1,W
961,Romania,Peru,1930,3,1,W
962,United States,Belgium,1930,3,0,W


In [293]:
away_matches

,opponent,team,year,goals_against,goals_for,result
0,Argentina,France,2022,3,3,D
1,Croatia,Morocco,2022,2,1,L
2,France,Morocco,2022,2,0,L
3,Argentina,Croatia,2022,3,0,L
4,Morocco,Portugal,2022,1,0,L
...,...,...,...,...,...,...
959,Argentina,France,1930,1,0,L
960,Yugoslavia,Brazil,1930,2,1,L
961,Romania,Peru,1930,3,1,L
962,United States,Belgium,1930,3,0,L


In [294]:
team_matches = pd.concat([home_matches, away_matches])

team_matches

,team,opponent,year,goals_for,goals_against,result
0,Argentina,France,2022,3,3,D
1,Croatia,Morocco,2022,2,1,W
2,France,Morocco,2022,2,0,W
3,Argentina,Croatia,2022,3,0,W
4,Morocco,Portugal,2022,1,0,W
...,...,...,...,...,...,...
959,France,Argentina,1930,0,1,L
960,Brazil,Yugoslavia,1930,1,2,L
961,Peru,Romania,1930,1,3,L
962,Belgium,United States,1930,0,3,L


In [295]:
team_summary = team_matches[[
    "team", 
    "result",
    "goals_for",
    "goals_against"
]].copy()

team_total = (
    team_summary
    .groupby(["team", "result"])
    .size()
    .unstack(fill_value=0)
    .reset_index() 
)

team_total["matches"] = (
    team_total["W"] + team_total["D"] + team_total["L"]
)

team_total["draw_rate"] = (
    team_total["D"]/team_total["matches"]
)

team_total["lost_rate"] = (
    team_total["L"]/team_total["matches"]
)

team_total["win_rate"] = (
    team_total["W"]/team_total["matches"]
)

team_total["total_point"] = (
    team_total["W"]*3 + team_total["D"]*1 + team_total["L"]*0
)

team_total["point_per_match"] = (
    team_total["total_point"]/team_total["matches"]
)

team_total.head()

result,team,D,L,W,matches,draw_rate,lost_rate,win_rate,total_point,point_per_match
0,Algeria,3,7,3,13,0.230769,0.538462,0.230769,12,0.923077
1,Angola,2,1,0,3,0.666667,0.333333,0.000000,2,0.666667
2,Argentina,17,24,47,88,0.193182,0.272727,0.534091,158,1.795455
3,Australia,4,12,4,20,0.200000,0.600000,0.200000,16,0.800000
4,Austria,4,13,12,29,0.137931,0.448276,0.413793,40,1.379310


In [296]:
goal_total = (
    team_summary
    .groupby(["team"])
    .agg({
        "goals_for": "sum",
        "goals_against": "sum"
    })
    .reset_index() 
)

goal_total["goals_difference"] = (
    goal_total["goals_for"] - goal_total["goals_against"]
)

goal_total

,team,goals_for,goals_against,goals_difference
0,Algeria,13,19,-6
1,Angola,1,2,-1
2,Argentina,152,101,51
3,Australia,17,37,-20
4,Austria,43,47,-4
...,...,...,...,...
81,Uruguay,89,76,13
82,Wales,5,10,-5
83,West Germany,106,63,43
84,Yugoslavia,55,42,13


In [297]:
team_total = team_total.merge(
    goal_total,
    on="team",
    how="left"
)

team_total["goal_per_matches"] = (
    team_total["goals_for"]/team_total["matches"]
)

team_total["against_per_matches"] = (
    team_total["goals_against"]/team_total["matches"]
)

team_total.head()

,team,D,L,W,matches,draw_rate,lost_rate,win_rate,total_point,point_per_match,goals_for,goals_against,goals_difference,goal_per_matches,against_per_matches
0,Algeria,3,7,3,13,0.230769,0.538462,0.230769,12,0.923077,13,19,-6,1.000000,1.461538
1,Angola,2,1,0,3,0.666667,0.333333,0.000000,2,0.666667,1,2,-1,0.333333,0.666667
2,Argentina,17,24,47,88,0.193182,0.272727,0.534091,158,1.795455,152,101,51,1.727273,1.147727
3,Australia,4,12,4,20,0.200000,0.600000,0.200000,16,0.800000,17,37,-20,0.850000,1.850000
4,Austria,4,13,12,29,0.137931,0.448276,0.413793,40,1.379310,43,47,-4,1.482759,1.620690


In [298]:
team_total = team_total.sort_values(
    "matches",
    ascending=False,
    ignore_index=True
)

team_total

,team,D,L,W,matches,draw_rate,lost_rate,win_rate,total_point,point_per_match,goals_for,goals_against,goals_difference,goal_per_matches,against_per_matches
0,Brazil,19,19,76,114,0.166667,0.166667,0.666667,247,2.166667,237,108,129,2.078947,0.947368
1,Argentina,17,24,47,88,0.193182,0.272727,0.534091,158,1.795455,152,101,51,1.727273,1.147727
2,Italy,21,17,45,83,0.253012,0.204819,0.542169,156,1.879518,128,77,51,1.542169,0.927711
3,England,22,20,32,74,0.297297,0.270270,0.432432,118,1.594595,104,68,36,1.405405,0.918919
4,France,14,20,39,73,0.191781,0.273973,0.534247,131,1.794521,136,85,51,1.863014,1.164384
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,Jamaica,0,2,1,3,0.000000,0.666667,0.333333,3,1.000000,3,9,-6,1.000000,3.000000
82,Angola,2,1,0,3,0.666667,0.333333,0.000000,2,0.666667,1,2,-1,0.333333,0.666667
83,Kuwait,1,2,0,3,0.333333,0.666667,0.000000,1,0.333333,2,6,-4,0.666667,2.000000
84,Zaire,0,3,0,3,0.000000,1.000000,0.000000,0,0.000000,0,14,-14,0.000000,4.666667
